# Phase 4a — C1, the optic-disc / fovea regressor

Trains **M2b**: ResNet18 to four coordinates, `[od_x, od_y, fovea_x, fovea_y]`.

This is the first piece of the evidence pathway, and everything else in Phase 4 sits on
it. Without disc and fovea positions there is no coordinate frame, and "haemorrhages in
three quadrants" — the 4-2-1 rule M3 reasons with — is undefined.

## The gate

C1 passes at a **mean landmark error below 0.5 disc diameters**. Below it, quadrant
assignment is reliable enough to reason over. Above it, M3 falls back to a count-only
rule — which `docs/00_START_HERE.md` names as the designed fallback, not a failure.

Error is measured in disc diameters rather than pixels because a fixed pixel tolerance
means different things at different fields of view. IDRiD publishes no disc diameter, so
it is derived per image from the disc-to-fovea distance ÷ 2.5, the standard clinical
relation.

## Before it can run: the Part A / Part C split

IDRiD ships **Part A** (81 images, lesion masks, numbered `IDRiD_01`–`81`) and
**Part B/C** (516 images, grades and centre coordinates, numbered `IDRiD_001`–`516`).
They are *different image sets*.

If your cache holds only Part A, the Part C coordinates have no image to join to and C1
has no targets. **Section 5 checks this before any GPU time is spent** and tells you
exactly what to do about it.

## Notebook settings

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Persistence | **Files only** |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

Inputs: `verify-dr-cache-512`, `verify-dr-idrid-masks`, and the Phase 2 manifests.

## Cost

Small. At most 516 images for 40 epochs is roughly **10-15 GPU-minutes** — the cheapest
experiment in the project.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to 'GPU T4 x2', then re-run from the top.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB")

## 3 · Helpers

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

## 4 · Find the cache and the manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()

if not ROOTS:
    raise RuntimeError(
        "No cache found. Attach verify-dr-cache-512 (and verify-dr-idrid-masks) "
        "under Add Data in the right-hand panel.")
if MANIFEST_DIR is None:
    raise RuntimeError(
        "No manifests found. Attach the Phase 2 output (verify-dr-manifests), or "
        "add 02_manifests.ipynb as a notebook input.")

DATASETS = resolve_datasets(ROOTS)
print("cache roots  :")
for r in ROOTS:
    print("   ", r)
print("manifest dir :", MANIFEST_DIR)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print("\ndatasets in the cache:")
print(f"  {'dataset':<12}{'images':>8}  {'masks':>5}  root")
for name, root in sorted(DATASETS.items()):
    d = root / name
    images = d / 'images'
    # rglob, not glob: glob('*') counts direct children only, so a nested
    # layout reports 1 and looks like catastrophic data loss when nothing
    # is actually wrong.
    n_img = sum(1 for f in images.rglob('*') if f.suffix.lower() in IMAGE_SUFFIXES) \
        if images.is_dir() else 0
    n_msk = len([m for m in (d / 'masks').iterdir() if m.is_dir()]) \
        if (d / 'masks').is_dir() else 0
    print(f"  {name:<12}{n_img:>8}  {n_msk:>5}  {root}")

    if images.is_dir():
        subdirs = [x for x in images.glob('*') if x.is_dir()]
        if subdirs and n_img:
            print(f"               ^ nested under {len(subdirs)} subdirectorie(s), "
                  f"e.g. {subdirs[0].name}/ - fine, the manifest stores full paths")
    if n_img == 0:
        print(f"               ^ NO IMAGES - {name} is empty in every attached root")

print("\nmanifests available:")
for c in sorted(MANIFEST_DIR.glob("*.csv")):
    print("  ", c.name)

print("\nMasks are a Phase 4 concern. M1 grades whole images and reads images +")
print("grades only, so 0 mask channels here blocks nothing in Phase 3. What")
print("matters now is that the datasets your chosen manifest names have images.")

# Every root goes to --cache-root, so each dataset resolves to the root that
# actually holds it instead of all of them being forced onto one.
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

## 5 · Can C1 actually run?

This cell fails loudly rather than letting a GPU session discover the problem. It checks
that an IDRiD manifest exists, that it carries coordinate columns, and that at least one
cached image actually has both centres.

In [ ]:
import pandas as pd

CANDIDATES = ['idrid_manifest.csv', 'idrid.csv']
GEOM_COLS = ['od_x', 'od_y', 'fovea_x', 'fovea_y']

MANIFEST = next((MANIFEST_DIR / n for n in CANDIDATES if (MANIFEST_DIR / n).exists()), None)
if MANIFEST is None:
    raise RuntimeError(
        f'No IDRiD manifest in {MANIFEST_DIR}. Looked for {CANDIDATES}. '
        'Re-run 02_manifests.ipynb with the IDRiD raw dataset attached.')

frame = pd.read_csv(MANIFEST)
print(f'{MANIFEST.name}: {len(frame)} rows')
print('columns:', ', '.join(frame.columns))

missing = [c for c in GEOM_COLS if c not in frame.columns]
if missing:
    raise RuntimeError(
        f'\n{MANIFEST.name} has no coordinate columns (missing {missing}).\n'
        'C1 has no training targets.\n\n'
        'Cause: prepare_manifest.py was run without --coords, or the Part C\n'
        'tables were not found. Re-run 02_manifests.ipynb with the IDRiD raw\n'
        'dataset attached and check its output for the line beginning \"!! No\n'
        'Part C coordinates\".')

usable = frame.dropna(subset=GEOM_COLS)
for flag in ('od_in_frame', 'fovea_in_frame'):
    if flag in usable.columns:
        usable = usable[usable[flag].astype(int) == 1]

print(f'\nrows with both centres in frame: {len(usable)} of {len(frame)}')

if len(usable) == 0:
    stems = sorted({__import__('pathlib').Path(p).stem for p in frame['image_path']})
    raise RuntimeError(
        '\nNo cached IDRiD image carries both centres, so C1 cannot run.\n\n'
        f'The cache holds {len(frame)} IDRiD images, named like: '
        f'{stems[:3]} ... {stems[-2:]}\n\n'
        'This is the Part A / Part C split. IDRiD Part A is 81 images with lesion\n'
        'masks, numbered IDRiD_01-81. Part C coordinates cover the 516 Part B\n'
        'grading images, numbered IDRiD_001-516 -- a different image set. If the\n'
        'cache holds only Part A, no stem joins and the coordinates attach to\n'
        'nothing.\n\n'
        'To fix: cache IDRiD Part B images (Phase 1, a CPU job, ~10 min for 516\n'
        'images), then re-run Phase 2 so the coordinates have something to join\n'
        'to. Part A stays as it is -- it is what C2 needs for masks.')

if len(usable) < 100:
    print(f'\nNOTE: only {len(usable)} usable images. docs/03 expects ~516 from Part C.')
    print('C1 can train on this, but treat the result as provisional -- a 0.5 DD')
    print('gate decided on a handful of validation images is not a decision.')

print()
print('grade column:', 'present' if 'grade' in frame.columns else 'absent')
if 'grade' in frame.columns and (frame['grade'] == -1).all():
    print('  all -1, i.e. masks/geometry only. Expected for Part A; C1 does not use grades.')

## 6 · Train

In [ ]:
EXPERIMENT = 'C1_geometry'

flags = [
    f"python {q(REPO_DIR / 'scripts/train_geometry.py')}",
    f'--manifest {q(MANIFEST)}',
    f'--experiment {q(EXPERIMENT)}',
    f'--results-dir {q(RESULTS)}',
    f'--cache-root {CACHE_FLAGS}',
    '--image-size 512',
    '--batch-size 16',
    '--epochs 40',
    '--lr 3e-4',
    '--val-frac 0.2',
    '--patience 10',
    '--workers 2',
    '--resume',
]
run(' '.join(flags))

## 7 · The C1 verdict

In [ ]:
m = json.loads((RESULTS / EXPERIMENT / 'metrics.json').read_text())
best, base = m['best_val'], m['baseline']
lift = base['mean_error_dd'] - m['best_mean_error_dd']
gate = 0.5

print(f"{m['experiment']}   best epoch {m['best_epoch']} of {m['epochs_run']}   "
      f"{m['minutes']:.0f} min   {m['train_images']} train / {m['val_images']} val")
print(f"  mean landmark error   {m['best_mean_error_dd']:.3f} disc diameters")
print(f"  optic disc            {best['od_error_px']:.1f} px   ({best['od_error_dd']:.3f} DD)")
print(f"  fovea                 {best['fovea_error_px']:.1f} px   ({best['fovea_error_dd']:.3f} DD)")
print(f"  within 0.5 DD         {best['within_half_dd'] * 100:.1f}% of landmarks")
print(f"  mean disc diameter    {best['mean_disc_diameter_px']:.1f} px")

print()
print(f"  constant-predictor baseline: {base['mean_error_dd']:.3f} DD")
print(f"  improvement over it:         {lift:+.3f} DD")

print()
print('=' * 72)
if m['best_mean_error_dd'] < gate:
    print(f"C1 PASSES: {m['best_mean_error_dd']:.3f} DD < {gate} gate.")
    print('Quadrant assignment is reliable enough for M3 to reason over, so the')
    print('partial 4-2-1 rule is available. C2 can proceed.')
else:
    print(f"C1 FAILS: {m['best_mean_error_dd']:.3f} DD is at or above the {gate} gate.")
    print('M3 must fall back to a count-only rule. docs/00_START_HERE.md names this')
    print('as the designed fallback, not a failure to hide -- record it and move on.')
if lift <= 0:
    print()
    print('WARNING: no better than predicting the training mean. The model learned')
    print('         the average fundus layout, not this image-s landmarks.')
print('=' * 72)

---
## 8 · Save

**Save Version → Save & Run All (Commit).** Publish `/kaggle/working/results` as
**`verify-dr-stage-c`** — `train_evidence.py` will load this encoder for C2, so the
checkpoint has to outlive the session.

Then record C1 in `docs/04_experiment_register.md`: the mean error in disc diameters,
both per-landmark figures, the margin over the constant baseline, and whether the gate
passed. If it failed, record that too and note that M3 drops to the count-only rule —
that is a documented design branch, not a result to bury.